# PRATYAKSHA · Step 3 — R3: give the AI the right information, computed

**What Step 2 showed:** small models cannot add or subtract, so they listed every invoice instead of choosing.
**What R3 changes:** code computes which invoices explain the amount (exactly, minus TDS, or slightly short),
adds the payment e-mails and any identical earlier credit, and the AI just **picks one option**.

**The fair test:** a no-AI *control* gets the same options and always picks the first one.
The AI earns its place only if it beats that control.

Run top to bottom. Runtime: **T4 GPU**. Upload `pratyaksha_step3.zip` to *Colab Notebooks* first.

## 1. Connect Drive
**Why:** code and results live in Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Unpack Step 3 code
**Why:** adds R3 without touching earlier results.

In [ ]:
%cd "/content/drive/MyDrive/Colab Notebooks"
!unzip -o -q pratyaksha_step3.zip
%cd "/content/drive/MyDrive/Colab Notebooks/pratyaksha"
!ls pratyaksha/systems

## 3. Install
**Why:** a new Colab session starts empty.

In [ ]:
!pip install -q -U transformers accelerate sentence-transformers rank-bm25 pytest

## 4. Data and tests
**Expect:** `26 passed`. **Why:** checks that the computed options never use future invoices and always add up.

In [ ]:
!python -m pratyaksha.world.generate --out data/wwi_in
!python -m pytest -q

## 5. The no-AI control (R3 context, always pick the first option)
**Why:** this is the bar the AI must beat. It uses the real reranker (GPU), but no language model. ~2 min.
**Watch:** *Right answer among computed options* = the best any model could do with these options.

In [ ]:
for cond in ["clean", "stress"]:
    !python -m pratyaksha.harness.runner --system r3_control --split dev --condition {cond} --cost-mode gpu

## 6. R1 with the 2B model (fair R1 vs R2 comparison)
**Why:** in Step 2 the 0.8B model scored ~0% in both R1 and R2, so we could not tell whether reranking helps.
2B works, so we run R1 at 2B and compare with the R2-2B result from Step 2. ~6 min each.

In [ ]:
for cond in ["clean", "stress"]:
    !python -m pratyaksha.harness.runner --system r1 --split dev --condition {cond} --cost-mode gpu \
        --opt model=Qwen/Qwen3.5-2B --opt batch=8 --opt k=5

## 7. R3 with the 0.8B model
**Why:** with the maths done for it, can even the smallest model now work?

In [ ]:
for cond in ["clean", "stress"]:
    !python -m pratyaksha.harness.runner --system r3 --split dev --condition {cond} --cost-mode gpu \
        --opt model=Qwen/Qwen3.5-0.8B --opt batch=8

## 8. R3 with the 2B model
**Why:** same question at the size that worked in Step 2.

In [ ]:
for cond in ["clean", "stress"]:
    !python -m pratyaksha.harness.runner --system r3 --split dev --condition {cond} --cost-mode gpu \
        --opt model=Qwen/Qwen3.5-2B --opt batch=8

## 9. (Optional) R3 with the 4B model
**Why:** does a bigger model read the e-mails better (bank-detail changes, which invoices)? ~8 GB download.
Batch 4 to fit memory.

In [ ]:
for cond in ["clean", "stress"]:
    !python -m pratyaksha.harness.runner --system r3 --split dev --condition {cond} --cost-mode gpu \
        --opt model=Qwen/Qwen3.5-4B --opt batch=4

## 10. Compare all runs
**Why:** this table is the Step 3 result.

In [ ]:
!python scripts/compare.py --split dev

## 11. What did the AI add over the control?
**Why:** shows the exact lines where the AI fixed or broke the control's answer. This is the evidence
for "what the model buys". (Edit `R3_Qwen3.5-2B` to `R3_Qwen3.5-4B` if you ran cell 9.)

In [ ]:
for cond in ["clean", "stress"]:
    !python scripts/vs_control.py --model R3_Qwen3.5-2B --condition {cond}

## 12. Send back
Download the notebook (*File → Download → .ipynb*) and send it, as you did for Step 2.